# Can a Bayesian MMM recover the truth? A parameter-recovery test

A marketing mix model is only useful if it can find effects that are really there.
With real data we never know the true answer, so the honest first test is to **simulate
data where we set the truth ourselves**, fit the model, and check what comes back.

This notebook does exactly that:

1. Simulate 3 years of weekly sales driven by two channels (Paid Social and TV) with
   known **carryover (adstock)** and **diminishing returns (saturation)**, plus seasonality,
   a promo and noise.
2. Fit a Bayesian MMM with [PyMC-Marketing](https://github.com/pymc-labs/pymc-marketing).
3. Check convergence diagnostics (R-hat, ESS) before trusting anything.
4. Compare recovered parameters, channel contributions and ROAS against the truth.

*Setup:* `pip install -r ../requirements.txt`

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pymc_marketing.mmm import MMM, GeometricAdstock, LogisticSaturation

SEED = 20260923
rng = np.random.default_rng(SEED)

# Chart style: one hue per channel, fixed order; truth drawn in dark ink
COLORS = {"social": "#2a78d6", "tv": "#eb6834"}
LABELS = {"social": "Paid Social", "tv": "TV"}
INK = "#1f1f1e"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5df", "grid.linewidth": 0.8,
    "axes.edgecolor": "#9a998f", "axes.labelcolor": "#4a4a45",
    "xtick.color": "#4a4a45", "ytick.color": "#4a4a45", "lines.linewidth": 2,
})

## 1. The ground truth

Two transformations sit at the heart of every MMM:

- **Geometric adstock** spreads a week's spend over the following weeks. With decay `alpha`,
  the weight on lag *l* is proportional to `alpha^l`. TV usually carries over longer than social.
- **Logistic saturation** `(1 - e^(-lam·x)) / (1 + e^(-lam·x))` bends the response curve so each
  extra rupee buys less. A larger `lam` means the channel saturates sooner.

These are the values the model has to find:

In [ ]:
TRUE = {
    "adstock_alpha":  {"social": 0.30, "tv": 0.60},
    "saturation_lam": {"social": 4.0,  "tv": 2.0},
    "effect_size":    {"social": 3000, "tv": 4500},   # max weekly sales a channel can add
}
L_MAX = 8

def geometric_adstock(x, alpha, l_max=L_MAX):
    w = alpha ** np.arange(l_max)
    w = w / w.sum()                      # normalised weights, as in PyMC-Marketing
    return np.convolve(x, w)[: len(x)]

def logistic_saturation(x, lam):
    return (1 - np.exp(-lam * x)) / (1 + np.exp(-lam * x))

pd.DataFrame(TRUE).rename(index=LABELS)

## 2. Simulate three years of weekly data

In [ ]:
n = 156
dates = pd.date_range("2022-01-03", periods=n, freq="W-MON")
t = np.arange(n)

# Paid Social: always on, with occasional bursts
social = rng.uniform(0.2, 0.6, n)
social[rng.random(n) > 0.85] += 0.5
social = social / social.max()

# TV: flighted, four to six weeks on, then dark
tv = np.zeros(n)
for start in range(0, n, 13):
    tv[start : start + rng.integers(4, 7)] = rng.uniform(0.5, 1.0)
tv = tv / tv.max()

spend = pd.DataFrame({"social": social * 1_500, "tv": tv * 3_000}, index=dates)

contrib_true = pd.DataFrame({
    ch: TRUE["effect_size"][ch] * logistic_saturation(
        geometric_adstock(spend[ch].values / spend[ch].max(), TRUE["adstock_alpha"][ch]),
        TRUE["saturation_lam"][ch])
    for ch in ["social", "tv"]
}, index=dates)

promo = np.zeros(n); promo[[40, 92, 144]] = 1
baseline = 10_000
season = 900 * np.sin(2 * np.pi * t / 52.18) + 400 * np.cos(2 * np.pi * t / 52.18)
noise = rng.normal(0, 350, n)

sales = baseline + season + 2_500 * promo + contrib_true.sum(axis=1).values + noise

df = pd.DataFrame({"date": dates, "social": spend["social"].values,
                   "tv": spend["tv"].values, "promo": promo, "sales": sales})
df.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True, layout="constrained")
axes[0].plot(df["date"], df["sales"], color=INK)
axes[0].set_ylabel("Sales")
for ax, ch in zip(axes[1:], ["social", "tv"]):
    ax.bar(df["date"], df[ch], width=5, color=COLORS[ch])
    ax.set_ylabel(f"{LABELS[ch]} spend")
fig.suptitle("Simulated weekly sales and media spend", x=0.01, ha="left", fontsize=13)
plt.show()

## 3. Fit the Bayesian MMM

The model knows the *form* of the transformations (geometric adstock, logistic saturation,
2 Fourier modes of yearly seasonality, one promo control) but nothing about the values.
Priors are PyMC-Marketing defaults, so the test is not rigged in the model's favour.

In [ ]:
mmm = MMM(
    date_column="date",
    channel_columns=["social", "tv"],
    control_columns=["promo"],
    adstock=GeometricAdstock(l_max=L_MAX),
    saturation=LogisticSaturation(),
    yearly_seasonality=2,
)

X, y = df.drop(columns="sales"), df["sales"]
_ = mmm.fit(X, y, chains=4, cores=2, draws=1000, tune=1500, target_accept=0.95,
            random_seed=SEED, progressbar=False)

## 4. Check convergence before reading any result

A posterior is only worth interpreting if the chains agree. Rules of thumb:
**R-hat ≤ 1.01** and **bulk ESS ≥ 400**. Divergent transitions should be zero or very close to it.

In [ ]:
summary = az.summary(mmm.idata, var_names=["adstock_alpha", "saturation_lam", "saturation_beta"],
                     kind="all", hdi_prob=0.94)
divergences = int(mmm.idata.sample_stats["diverging"].sum())
print(f"Divergent transitions: {divergences}")
print(f"Max R-hat: {summary['r_hat'].max():.3f} | Min bulk ESS: {summary['ess_bulk'].min():.0f}")
summary[["mean", "sd", "hdi_3%", "hdi_97%", "r_hat", "ess_bulk"]].round(3)

## 5. Did the model recover the true parameters?

In [ ]:
post = mmm.idata.posterior
rows = []
for param in ["adstock_alpha", "saturation_lam"]:
    for ch in ["social", "tv"]:
        draws = post[param].sel(channel=ch).values.ravel()
        lo, hi = az.hdi(draws, hdi_prob=0.94)
        truth = TRUE[param][ch]
        rows.append({"parameter": param, "channel": LABELS[ch], "true": truth,
                     "posterior mean": draws.mean(), "94% HDI low": lo, "94% HDI high": hi,
                     "truth inside HDI": lo <= truth <= hi})
recovery = pd.DataFrame(rows)
recovery.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), layout="constrained")
for ax, param, title in zip(axes, ["adstock_alpha", "saturation_lam"],
                            ["Adstock decay (alpha)", "Saturation speed (lam)"]):
    sub = recovery[recovery["parameter"] == param].reset_index(drop=True)
    for i, r in sub.iterrows():
        ch = "social" if r["channel"] == "Paid Social" else "tv"
        ax.hlines(i, r["94% HDI low"], r["94% HDI high"], color=COLORS[ch], lw=6, alpha=0.35)
        ax.plot(r["posterior mean"], i, "o", color=COLORS[ch], ms=9, label="Posterior mean ± 94% HDI" if i == 0 else None)
        ax.plot(r["true"], i, "|", color=INK, ms=22, mew=2.5, label="True value" if i == 0 else None)
    ax.set_yticks(range(len(sub)), sub["channel"])
    ax.set_ylim(-0.6, len(sub) - 0.4)
    ax.set_title(title, loc="left", fontsize=11)
axes[0].legend(loc="lower right", frameon=False, fontsize=9)
plt.show()

## 6. Contributions and ROAS: the numbers a CMO actually uses

Parameters are a means to an end. What the business acts on is **how much each channel
contributed** and **what it returned per unit of spend**. Here we compare both, on the
original sales scale, against the truth.

In [ ]:
contrib_post = mmm.compute_channel_contribution_original_scale()   # (chain, draw, date, channel)
total_post = contrib_post.sum("date")

rows = []
for ch in ["social", "tv"]:
    draws = total_post.sel(channel=ch).values.ravel()
    lo, hi = az.hdi(draws, hdi_prob=0.94)
    s = df[ch].sum()
    true_total = contrib_true[ch].sum()
    rows.append({"channel": LABELS[ch],
                 "true ROAS": true_total / s,
                 "estimated ROAS": draws.mean() / s,
                 "ROAS 94% HDI": f"{lo / s:.2f} – {hi / s:.2f}",
                 "error vs truth": f"{(draws.mean() - true_total) / true_total:+.1%}"})
pd.DataFrame(rows).round(2)

In [ ]:
mean_ts = contrib_post.mean(("chain", "draw"))
hdi_ts = az.hdi(contrib_post.to_dataset(name="c"), hdi_prob=0.94)["c"]

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, layout="constrained")
for ax, ch in zip(axes, ["social", "tv"]):
    band = hdi_ts.sel(channel=ch)
    ax.fill_between(df["date"], band.sel(hdi="lower"), band.sel(hdi="higher"),
                    color=COLORS[ch], alpha=0.25, lw=0, label="94% HDI")
    ax.plot(df["date"], mean_ts.sel(channel=ch), color=COLORS[ch], label="Estimated")
    ax.plot(df["date"], contrib_true[ch], color=INK, lw=1.4, ls="--", label="True")
    ax.set_title(f"{LABELS[ch]}: weekly contribution to sales", loc="left", fontsize=11)
axes[0].legend(frameon=False, ncol=3, loc="upper right", fontsize=9)
plt.show()

**Read the two channels differently.** TV runs in on/off flights, so the data contains sharp
contrasts and the model pins its effect down tightly. Paid Social is always on with little
variation, so its interval is several times wider even though the point estimate lands near the
truth. Same model, same data volume: the *spend pattern* decides how much you can learn.
That is a media-planning argument for deliberately varying spend, or running a holdout test.

## 7. What this shows, and what it doesn't

**Shows**
- With the right functional form and enough variation in spend, a Bayesian MMM recovers
  carryover, saturation and ROAS, and its uncertainty intervals are honest about what it can't pin down.
- Diagnostics come first: a model that hasn't converged can print confident, wrong ROAS numbers.

**Doesn't show**
- Real data breaks the assumptions used here: channels move together, spend follows demand
  (endogeneity), and the true functional form is unknown. Recovery on clean simulated data is a
  *necessary* test, not a *sufficient* one.
- That is why production measurement pairs the MMM with **incrementality experiments**
  (geo-holdouts, lift tests) and uses them to calibrate the model.

**Try next:** make TV and Social spend correlated, or shrink the spend variation, and watch the
HDIs widen. That is the model telling you the data can't separate the channels.